In [2]:
!pip install pymysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 1.8 MB/s eta 0:00:00


In [5]:
import pymysql

# Connect to TiDB
connection = pymysql.connect(
    host='gateway01.ap-southeast-1.prod.aws.tidbcloud.com',
    port=4000,
    user='4JX5zZKcrMbbGKn.root',
    password='Yf2kqFJjt1Ms2ruT',
    database='students'
)

In [6]:
import pandas as pd

In [7]:
query = "SELECT * FROM stem;"
df = pd.read_sql(query, connection)
print("dataset shape:",df.shape)
df.head()

/tmp/ipykernel_5847/787082883.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


dataset shape: (30642, 15)


,column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,column_13,column_14,column_15
0,ID,Gender,EthnicGroup,ParentEduc,LunchType,TestPrep,ParentMaritalStatus,PracticeSport,IsFirstChild,NrSiblings,TransportMeans,WklyStudyHours,MathScore,ReadingScore,WritingScore
1,0,female,,bachelor's degree,standard,none,married,regularly,yes,3,school_bus,< 5,71,71,74
2,1,female,group C,some college,standard,,married,sometimes,yes,0,,05-Oct,69,90,88
3,2,female,group B,master's degree,standard,none,single,sometimes,yes,4,school_bus,< 5,87,93,91
4,3,male,group A,associate's degree,free/reduced,none,married,never,no,1,,05-Oct,45,56,42


In [9]:
df.columns = df.iloc[0]
df = df[1:].copy()


numeric_cols = ['NrSiblings', 'MathScore', 'ReadingScore', 'WritingScore']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

categorical_cols = ['Gender', 'EthnicGroup', 'ParentEduc', 'LunchType',
                    'TestPrep', 'ParentMaritalStatus', 'PracticeSport',
                    'IsFirstChild', 'TransportMeans', 'WklyStudyHours']

df[categorical_cols] = df[categorical_cols].fillna('Unknown')

# (replace missing siblings with median)
df['NrSiblings'] = df['NrSiblings'].fillna(df['NrSiblings'].median())

# (fill with median)
score_cols = ['MathScore', 'ReadingScore', 'WritingScore']
for col in score_cols:
    df[col] = df[col].fillna(df[col].median())

print("Remaining Null Values:")
print(df.isnull().sum())

Remaining Null Values:
0
ID                     0
Gender                 0
EthnicGroup            0
ParentEduc             0
LunchType              0
TestPrep               0
ParentMaritalStatus    0
PracticeSport          0
IsFirstChild           0
NrSiblings             0
TransportMeans         0
WklyStudyHours         0
MathScore              0
ReadingScore           0
WritingScore           0
dtype: int64


In [12]:
import statsmodels.api as sm
import statsmodels.formula.api as smf


df['OverallScore'] = (df['MathScore'] + df['ReadingScore'] + df['WritingScore']) / 3.0


# Target: MathScore (or OverallScore)
# Predictors: TestPrep, WklyStudyHours, ParentEduc, NrSiblings
model = smf.ols(
    'MathScore ~ C(TestPrep) + C(WklyStudyHours) + C(ParentEduc) + NrSiblings',
    data=df
).fit()

# 3. Print Statistical Model Results
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              MathScore   R-squared:                       0.062
Model:                            OLS   Adj. R-squared:                  0.062
Method:                 Least Squares   F-statistic:                     168.9
Date:                Mon, 31 Aug 2026   Prob (F-statistic):               0.00
Time:                        08:28:35   Log-Likelihood:            -1.2620e+05
No. Observations:               30641   AIC:                         2.524e+05
Df Residuals:                   30628   BIC:                         2.525e+05
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [13]:
df['PredictedMathScore'] = model.predict(df)
df['Residuals'] = df['MathScore'] - df['PredictedMathScore']

# Export cleaned data to CSV for Excel Dashboard
df.to_csv("cleaned_student_performance_output.csv", index=False)
print("File 'cleaned_student_performance_output.csv' successfully saved!")

File 'cleaned_student_performance_output.csv' successfully saved!
